# Schema exploration

Introspects `data/olist.db` through `fetch_all` (trusted PRAGMA calls) rather than raw `sqlite3`, so this notebook exercises the same code path `schema.py` will eventually use for schema retrieval. Produces: table/column inventory, foreign keys, indexes, and a compact rendered schema block of the kind that would go into an LLM prompt.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from db import fetch_all, get_connection
from config import settings

conn = get_connection()
tables = [r[0] for r in fetch_all(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
)]
tables

['customers',
 'geolocation',
 'order_items',
 'order_payments',
 'order_reviews',
 'orders',
 'product_category_translation',
 'products',
 'sellers']

In [2]:
# columns per table, via PRAGMA table_info
rows = []
for t in tables:
    for cid, name, coltype, notnull, default, pk in fetch_all(f"PRAGMA table_info({t})"):
        rows.append((t, name, coltype, bool(notnull), bool(pk)))

columns_df = pd.DataFrame(rows, columns=["table", "column", "type", "not_null", "primary_key"])
columns_df

,table,column,type,not_null,primary_key
0,customers,customer_id,TEXT,False,True
1,customers,customer_unique_id,TEXT,True,False
2,customers,customer_zip_code_prefix,TEXT,False,False
3,customers,customer_city,TEXT,False,False
4,customers,customer_state,TEXT,False,False
5,geolocation,geolocation_zip_code_prefix,TEXT,False,False
6,geolocation,geolocation_lat,REAL,False,False
7,geolocation,geolocation_lng,REAL,False,False
8,geolocation,geolocation_city,TEXT,False,False
9,geolocation,geolocation_state,TEXT,False,False


In [3]:
# foreign keys — this is the join graph the agent has to reason over
fk_rows = []
for t in tables:
    for _id, _seq, ref_table, frm, to, *_ in fetch_all(f"PRAGMA foreign_key_list({t})"):
        fk_rows.append((t, frm, ref_table, to))

fk_df = pd.DataFrame(fk_rows, columns=["table", "column", "references_table", "references_column"])
fk_df

,table,column,references_table,references_column
0,order_items,seller_id,sellers,seller_id
1,order_items,product_id,products,product_id
2,order_items,order_id,orders,order_id
3,order_payments,order_id,orders,order_id
4,order_reviews,order_id,orders,order_id
5,orders,customer_id,customers,customer_id
6,products,product_category_name,product_category_translation,product_category_name


In [4]:
# indexes per table — cross-check against what load_data.py declared
idx_rows = []
for t in tables:
    for _seq, name, unique, *_ in fetch_all(f"PRAGMA index_list({t})"):
        idx_rows.append((t, name, bool(unique)))

pd.DataFrame(idx_rows, columns=["table", "index", "unique"])

,table,index,unique
0,customers,idx_customers_customer_zip_code_prefix,False
1,customers,idx_customers_customer_state,False
2,customers,idx_customers_customer_unique_id,False
3,customers,sqlite_autoindex_customers_1,True
4,geolocation,idx_geolocation_geolocation_state,False
5,geolocation,idx_geolocation_geolocation_zip_code_prefix,False
6,order_items,idx_order_items_seller_id,False
7,order_items,idx_order_items_product_id,False
8,order_items,sqlite_autoindex_order_items_1,True
9,order_payments,idx_order_payments_payment_type,False


In [5]:
# sample values per column — mirrors settings.include_sample_values / sample_values_per_column,
# which is what the rendered schema block would show the model
def sample_values(table: str, column: str, n: int) -> list:
    rows = fetch_all(
        f"SELECT DISTINCT {column} FROM {table} WHERE {column} IS NOT NULL LIMIT {n}"
    )
    return [r[0] for r in rows]

preview_rows = []
for t, col in [("orders", "order_status"), ("order_payments", "payment_type"),
               ("customers", "customer_state"), ("products", "product_category_name")]:
    preview_rows.append((t, col, sample_values(t, col, settings.sample_values_per_column)))

pd.DataFrame(preview_rows, columns=["table", "column", "sample_values"])

,table,column,sample_values
0,orders,order_status,"[approved, canceled, created]"
1,order_payments,payment_type,"[boleto, credit_card, debit_card]"
2,customers,customer_state,"[AC, AL, AM]"
3,products,product_category_name,"[agro_industria_e_comercio, alimentos, aliment..."


In [6]:
# compact rendered schema, roughly what schema.py would hand to the prompt
def render_schema(tables: list[str]) -> str:
    lines = []
    for t in tables:
        cols = fetch_all(f"PRAGMA table_info({t})")
        col_strs = [f"{name} {coltype}" + (" PK" if pk else "") for _cid, name, coltype, _nn, _def, pk in cols]
        fks = fetch_all(f"PRAGMA foreign_key_list({t})")
        fk_strs = [f"{frm} -> {ref}({to})" for _id, _seq, ref, frm, to, *_ in fks]
        block = f"TABLE {t} (\n  " + ",\n  ".join(col_strs) + "\n)"
        if fk_strs:
            block += "\n  FOREIGN KEYS: " + "; ".join(fk_strs)
        lines.append(block)
    return "\n\n".join(lines)

print(render_schema(["orders", "order_items", "products"]))

TABLE orders (
  order_id TEXT PK,
  customer_id TEXT,
  order_status TEXT,
  order_purchase_timestamp TEXT,
  order_approved_at TEXT,
  order_delivered_carrier_date TEXT,
  order_delivered_customer_date TEXT,
  order_estimated_delivery_date TEXT
)
  FOREIGN KEYS: customer_id -> customers(customer_id)

TABLE order_items (
  order_id TEXT PK,
  order_item_id INTEGER PK,
  product_id TEXT,
  seller_id TEXT,
  shipping_limit_date TEXT,
  price REAL,
  freight_value REAL
)
  FOREIGN KEYS: seller_id -> sellers(seller_id); product_id -> products(product_id); order_id -> orders(order_id)

TABLE products (
  product_id TEXT PK,
  product_category_name TEXT,
  product_name_lenght INTEGER,
  product_description_lenght INTEGER,
  product_photos_qty INTEGER,
  product_weight_g INTEGER,
  product_length_cm INTEGER,
  product_height_cm INTEGER,
  product_width_cm INTEGER
)
  FOREIGN KEYS: product_category_name -> product_category_translation(product_category_name)
